# 3장 — 흔들리는 나침반: 미니배치 SGD 🧭

기울기가 표본 추정치라는 것(σ/√B)을 숫자로 확인하고, 노이즈가 여행자를 구하는
반전(GD vs SGD 경주)을 재현합니다.

**이 노트북의 목표**
1. 배치 크기별 기울기 추정의 흔들림이 정확히 1/√B 로 주는 것을 검증
2. 이중 골짜기에서 GD(갇힘) vs SGD(탈출) — 노이즈의 순기능 재현
3. 에포크·스텝 산수 — 설정 파일 읽기의 기본기

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch3')

## 1. 실험 ① — 흔들림은 정확히 σ/√B

데이터 하나하나의 기울기 = 진짜 기울기 + 노이즈(σ=0.9). 배치 평균의 산포를 실측해
이론(σ/√B)과 대조합니다.

> σ(읽기: **시그마**)는 **데이터별 기울기가 서로 얼마나 다른가** — 3부 2장의 그 **표준편차**입니다.
> 데이터가 서로 딴소리를 할수록(σ 큼) 나침반이 크게 흔들리고, 배치를 키울수록(√B) 잦아듭니다.

In [ ]:
import numpy as np
rng = np.random.default_rng(303)

TRUE_G, SIGMA = 1.0, 0.9

print(' 배치 B   흔들림(실측)   이론 σ/√B')
stds = {}
for B in [1, 4, 16, 64, 256]:
    batch_means = TRUE_G + SIGMA * rng.standard_normal((3000, B)).mean(axis=1)
    stds[B] = batch_means.std()
    print(f'  {B:5d}    {stds[B]:8.4f}      {SIGMA/np.sqrt(B):8.4f}')
    assert abs(stds[B] - SIGMA/np.sqrt(B)) < 0.1, '실측 ≈ 이론'

assert stds[4] < stds[1] and stds[64] < stds[16], '배치가 클수록 흔들림이 준다'
assert 1.7 < stds[16]/stds[64] < 2.3, 'B 4배 → 흔들림 절반 (√4 = 2)'
print('\n✔ 확인: 시리즈 네 번째 1/√n — 정확도는 √B로만 늘고 비용은 B로 는다 (수확 체감)')

## 2. 실험 ② — 노이즈의 순기능: GD vs SGD 경주

오른쪽 얕은 웅덩이(L ≈ 0.76) 위에서 출발 — 왼쪽에 더 깊은 골짜기(L ≈ −0.84)가 있습니다.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

L = lambda w: (w**2 - 4)**2 / 16 + 0.4*w      # 이중 골짜기 (기울어진)
g = lambda w: w*(w**2 - 4)/4 + 0.4

def gd(steps=150, lr=0.15):
    w, hist = 2.8, [2.8]
    for _ in range(steps):
        w -= lr * g(w)
        hist.append(w)
    return np.array(hist)

def sgd(steps=150, lr=0.15, noise=2.5, seed=3):
    r = np.random.default_rng(seed)
    w, hist = 2.8, [2.8]
    for _ in range(steps):
        w -= lr * (g(w) + noise * r.standard_normal())   # 기울기 + 미니배치 노이즈
        hist.append(w)
    return np.array(hist)

hg, hs = gd(), sgd()
ws = np.linspace(-3.4, 3.4, 300)
plt.figure(figsize=(7, 4))
plt.plot(ws, L(ws), 'k-', lw=1.5, label='손실 지형')
plt.plot(hg, L(hg), '.-', ms=3, color='tab:blue', label='🔵 GD (정확한 나침반)')
plt.plot(hs, L(hs), '.', ms=2.5, color='tab:orange', alpha=0.6, label='🟠 SGD (흔들리는 나침반)')
plt.scatter([-2.19], [L(-2.19)], marker='*', s=200, c='gold', edgecolors='k', zorder=5, label='깊은 골짜기 🚩')
plt.xlabel('w'); plt.ylabel('손실'); plt.legend(); plt.grid(alpha=0.3)
plt.title('같은 출발점, 다른 나침반 — 완벽함이 함정에 빠진다'); plt.tight_layout(); plt.show()

print(f'🔵 GD  최종: w = {hg[-1]:.2f}, L = {L(hg[-1]):.3f} (얕은 웅덩이)')
print(f'🟠 SGD 최종: w = {hs[-1]:.2f}, L = {L(hs[-1]):.3f} (깊은 골짜기!)')

assert abs(hg[-1] - 1.77) < 0.1, 'GD는 얕은 웅덩이(w≈1.77)에 갇힌다 — 기울기 0은 종착역'
assert hs[-1] < -1.5 and L(hs[-1]) < -0.5, 'SGD는 노이즈 덕에 언덕을 넘어 깊은 골짜기로'
print('\n✔ 확인: 흔들림 = 공짜 담금질 — 얕은 함정을 못 본 척 지나가게 해 준다 (시드 고정 — 재현 가능)')

## 3. 실험 ③ — 에포크·스텝 산수 (설정 파일 기본기)

In [ ]:
def total_steps(n_data, batch_size, num_epochs):
    """총 스텝 수 = (데이터 수 ÷ 배치 크기) × 에포크 수."""
    return (n_data // batch_size) * num_epochs

print(' 데이터 N   batch_size   epochs   총 스텝')
for n, b, e in [(1000, 50, 1), (100_000, 100, 3), (100_000, 400, 3)]:
    print(f'  {n:8,}   {b:8d}   {e:5d}   {total_steps(n, b, e):8,}')

assert total_steps(1000, 50, 1) == 20
assert total_steps(100_000, 100, 3) == 3000
assert total_steps(100_000, 400, 3) == 750, '배치를 4배 키우면 (에포크 고정 시) 스텝은 1/4'
print('\n✔ 확인: 배치를 키우면 스텝 수가 준다 — "배치만 바꿨는데 성능 하락"의 절반은 이 산수 미보정')

## 4. 관찰 정리 + 🤖 AI 연결

- 기울기 = 데이터별 기울기의 **평균** → 미니배치 B개의 평균은 **표본 추정치**,
  흔들림 ≈ **σ/√B** — 시리즈 네 번째 1/√n (3부 7장 큰 수의 법칙).
- **정확도는 √B, 비용은 B** — 수확 체감이라 적당히 흔들리는 중간 배치가 실전의 답.
- **노이즈의 순기능**: 완벽한 나침반(GD)은 얕은 웅덩이에 갇히지만, 흔들리는 나침반(SGD)은
  넘어간다 — 공짜 담금질. (단 정착기엔 방해 → 단계별 조절은 6장 스케줄.)
- 에포크 = 전체 한 바퀴, 스텝 = (N/B)×에포크 — 설정 파일(batch_size·num_epochs) 읽기의 기본.

**다음 챕터**: 4장 — 관성으로 달리기. 좁고 긴 골짜기에서 벽 사이를 지그재그로 튕기는 문제,
그 해법인 **모멘텀**(속도의 누적 = 지수이동평균)을 공 굴리듯 체험합니다.